# 01 — Dataset Exploration

Goal: load both datasets' real metadata, confirm (or correct) the label
mapping assumed in `src/manifest.py`, quantify class balance and audio
properties, and write out a combined manifest that Phase 4 training
will consume directly.

Run this **after** `src/download_coughvid.py` and `src/download_icbhi.py`
have finished.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf

from config import COUGHVID_RAW_DIR, ICBHI_RAW_DIR, FIGURES_DIR, DATA_PROCESSED_DIR
from manifest import build_coughvid_manifest, build_icbhi_manifest, assign_splits

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.figsize"] = (7, 4)

## 1. COUGHVID — load & inspect metadata

In [ ]:
# NOTE: if this errors on a KeyError, open public_dataset.csv yourself and
# check the real column names — dataset versions have shifted these before.
# Update the *_COLUMNS constants / this cell to match, then re-run.
raw_coughvid = pd.read_csv(COUGHVID_RAW_DIR / "public_dataset.csv")
print(raw_coughvid.shape)
raw_coughvid.head()

In [ ]:
print("Missing values per column:")
print(raw_coughvid.isna().mean().sort_values(ascending=False).head(15))

print("\ndiagnosis value counts (pulmonologist-labeled subset):")
print(raw_coughvid["diagnosis"].value_counts(dropna=False))

## 2. ICBHI — load & inspect metadata

In [ ]:
diag = pd.read_csv(ICBHI_RAW_DIR / "patient_diagnosis.csv", header=None,
                    names=["patient_id", "diagnosis"])
print(diag.shape)
print(diag["diagnosis"].value_counts())

n_txt = len(list(ICBHI_RAW_DIR.glob("*.txt")))
n_wav = len(list(ICBHI_RAW_DIR.glob("*.wav")))
print(f"\nannotation files: {n_txt}, wav files: {n_wav}")

## 3. Build manifests with the common label target

Default target: **normal vs abnormal** (see docstring in `src/manifest.py`
for exactly which diagnoses map to which label, and why this is the most
defensible common ground between the two label schemes). Re-run the cells
above first if any KeyErrors showed up — the mapping must match your
actual downloaded files before this step is trustworthy.

In [ ]:
coughvid_df = build_coughvid_manifest()
coughvid_df = assign_splits(coughvid_df, group_col=None, train_frac=0.85)
coughvid_df["dataset"] = "coughvid"
coughvid_df.head()

In [ ]:
icbhi_df = build_icbhi_manifest()
# ICBHI is the frozen EXTERNAL test set end-to-end (Phase completion rule:
# never used to tune the model) -> every row gets split="test", regardless
# of the random assign_splits helper used for COUGHVID's train/val.
icbhi_df["split"] = "test"
icbhi_df["dataset"] = "icbhi"
icbhi_df.head()

## 4. Class distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
coughvid_df["label"].value_counts().plot.bar(ax=axes[0], title="COUGHVID (train/val)")
icbhi_df["label"].value_counts().plot.bar(ax=axes[1], title="ICBHI (external test)")
for ax in axes:
    ax.set_xlabel("label"); ax.set_ylabel("count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_distribution.png", dpi=150)
plt.show()

print("COUGHVID class balance:\n", coughvid_df["label"].value_counts(normalize=True))
print("\nICBHI class balance:\n", icbhi_df["label"].value_counts(normalize=True))

## 5. Audio duration distribution

In [ ]:
def get_duration(path):
    try:
        info = sf.info(path)
        return info.frames / info.samplerate
    except Exception:
        return np.nan

sample_n = min(500, len(coughvid_df))
durations = coughvid_df["filepath"].sample(sample_n, random_state=42).apply(get_duration)
durations.dropna().hist(bins=40)
plt.title(f"COUGHVID clip durations (n={sample_n} sample)")
plt.xlabel("seconds"); plt.ylabel("count")
plt.savefig(FIGURES_DIR / "coughvid_duration_hist.png", dpi=150)
plt.show()
print(durations.describe())

In [ ]:
icbhi_cycle_len = icbhi_df["end_s"] - icbhi_df["start_s"]
icbhi_cycle_len.hist(bins=40)
plt.title("ICBHI respiratory-cycle durations")
plt.xlabel("seconds"); plt.ylabel("count")
plt.savefig(FIGURES_DIR / "icbhi_cycle_duration_hist.png", dpi=150)
plt.show()
print(icbhi_cycle_len.describe())

## 6. Listen to / look at a few representative waveforms

Sanity-check that loading actually works and clips aren't silent/corrupted
before committing to a preprocessing config.

In [ ]:
import librosa, librosa.display

sample_rows = coughvid_df.groupby("label").sample(1, random_state=42)
fig, axes = plt.subplots(len(sample_rows), 1, figsize=(9, 3*len(sample_rows)))
for ax, (_, row) in zip(np.atleast_1d(axes), sample_rows.iterrows()):
    y, sr = librosa.load(row["filepath"], sr=None)
    ax.plot(y)
    ax.set_title(f'{row["label"]} — {row["filepath"]}')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "sample_waveforms.png", dpi=150)
plt.show()

## 7. Data-quality findings (fill in after running the cells above)

- Corrupted / zero-length files found: **TODO**
- Any diagnosis categories not covered by the mapping in `manifest.py`: **TODO**
- Class imbalance severity (train vs external test): **TODO — feeds Phase 5 augmentation/class-weight decision**
- Final decision on prediction target: **normal vs abnormal, confirmed / revised** — TODO

## 8. Save the combined manifest for training

In [ ]:
combined = pd.concat([coughvid_df, icbhi_df], ignore_index=True, sort=False)
out_path = DATA_PROCESSED_DIR / "manifest.csv"
combined.to_csv(out_path, index=False)
print(f"saved {len(combined)} rows -> {out_path}")
combined["split"].value_counts()